In [77]:
import pandas as pd
from pathlib import Path

In [ ]:
# 1. Define Paths Robustly (Works in both Jupyter and standard scripts)
current_dir = Path.cwd()

# Handle path depending on whether you run this from root or inside the 'scripts' folder
if current_dir.name == "scripts":
    root_dir = current_dir.parent
else:
    root_dir = current_dir

seller_data_dir = root_dir / 'data' / 'csvs'
seller_name = 'Klassy'
data_file = f"{seller_name}_products.csv"
data_file_path = seller_data_dir / data_file


# Upload directory
upload_dir = root_dir / 'data' / 'csvs' / 'uploads' / seller_name
upload_dir.mkdir(parents=True, exist_ok=True)


In [79]:
# Define some helper functions for cleaning
def slugify(text):
    if isinstance(text, str):
        # Remove anything that is not alphanumeric or space, then replace spaces with hyphens and convert to lowercase
        text = ''.join(e for e in text if e.isalnum() or e.isspace())
        text = text.replace(' ', '-').lower()
    return text

In [80]:
df = pd.read_csv(data_file_path)

# Remove any 'Unnamed' columns caused by trailing commas in the CSV
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# keep='last' retains the last instance it finds and drops previous duplicates
df = df.drop_duplicates(subset=['seller_product_url'], keep='last')
print(f"\nTotal unique products after dropping duplicate links: {len(df)}")

print(f"Loaded {data_file_path} - Shape: {df.shape}")


Total unique products after dropping duplicate links: 11364
Loaded /home/abrar/Freelance/DaamDekho/Crawler/data/csvs/Ryans_products.csv - Shape: (11364, 35)


In [81]:
# Show the dataframe info
print(df.info())

<class 'pandas.DataFrame'>
Index: 11364 entries, 0 to 11535
Data columns (total 35 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   category_slug            11358 non-null  str    
 1   product_name             11364 non-null  str    
 2   product_slug             11364 non-null  str    
 3   seller_slug              0 non-null      float64
 4   current_price            11362 non-null  float64
 5   seller_product_url       11364 non-null  str    
 6   brand_slug               11358 non-null  str    
 7   brand_name               11358 non-null  str    
 8   product_description      11363 non-null  str    
 9   model                    0 non-null      float64
 10  sku                      0 non-null      float64
 11  primary_image_url        11355 non-null  str    
 12  image_urls               0 non-null      float64
 13  specifications           0 non-null      float64
 14  attributes               0 non-null   

# Tasks

- Inspect category_slug null issues
- Inspect category_name null issues
- Populate the seller_slug with "ryans"
- Inspect why some current_price is missing
- Inspect why some original_price is missing
- Inspect why some brand_slug and brand_name is missing (Also check the data situation of brand_name and brand_slug)
- Inspect primary_image_url being null

## category_slug and category_name inspection

In [82]:
# Check the category_name column for any anomalies
print("\nUnique categories in 'category_name':")
pd.set_option('display.max_rows', None)
print(df['category_name'].value_counts())
pd.reset_option('display.max_rows')


Unique categories in 'category_name':
category_name
Cable / Converter / HUB             580
Mouse                               438
Toner                               407
Headphone                           394
Casing                              393
Keyboard                            350
All Laptop                          266
Speaker                             264
Smartwatch                          256
Motherboard                         252
Earbuds                             226
CPU Cooler                          212
Internal SSD                        212
Smart Phone                         211
All Monitor                         200
All Brands                          182
Kitchen                             176
Microphone                          169
Network Router                      160
Network Switch                      158
Power Supply                        149
Network Storage                     148
Desktop Ram                         145
Gaming Monitor             

In [83]:
# Need to convert some category_name values to match expected categories
# Anything that starts with 'All', remove the 'All' prefix and strip whitespace
df['category_name'] = df['category_name'].apply(lambda x: x[3:].strip() if isinstance(x, str) and x.startswith('All') else x)

# Check the category_name column again after transformation
print("\nUnique categories in 'category_name' after transformation:")
pd.set_option('display.max_rows', None)
print(df['category_name'].value_counts())
pd.reset_option('display.max_rows')


Unique categories in 'category_name' after transformation:
category_name
Cable / Converter / HUB             580
Mouse                               438
Toner                               407
Headphone                           394
Casing                              393
Keyboard                            350
Laptop                              269
Speaker                             264
Smartwatch                          256
Motherboard                         252
Earbuds                             226
CPU Cooler                          212
Internal SSD                        212
Smart Phone                         211
Monitor                             201
Brands                              182
Kitchen                             176
Microphone                          169
Network Router                      160
Network Switch                      158
Power Supply                        149
Network Storage                     148
Desktop Ram                         145
Gaming

In [84]:

df['category_slug'] = df['category_name'].apply(slugify)
print("\nUnique category slugs generated:")
pd.set_option('display.max_rows', None)
print(df['category_slug'].value_counts())
pd.reset_option('display.max_rows')


Unique category slugs generated:
category_slug
cable--converter--hub               580
mouse                               438
toner                               407
headphone                           394
casing                              393
keyboard                            350
laptop                              269
speaker                             264
smartwatch                          256
motherboard                         252
earbuds                             226
cpu-cooler                          212
internal-ssd                        212
smart-phone                         211
monitor                             201
brands                              182
kitchen                             176
microphone                          169
network-router                      160
network-switch                      158
power-supply                        149
network-storage                     148
desktop-ram                         145
gaming-monitor                  

In [85]:
# Set pandas display options to show full URLs
pd.set_option('display.max_colwidth', None)

# Show the records with name and link to understand the context of nulls
print("\nRecords with null category_name or category_slug:")
print(df[df['category_name'].isnull() | df['category_slug'].isnull()][['product_name', 'seller_product_url', 'category_name', 'category_slug']])




Records with null category_name or category_slug:
                                                                               product_name  \
11354                               Microsoft Visual Studio Pro 2019 SNGL OLP NL #C5E-01380   
11385            UGREEN AV119 (10729) 3.5mm Male to Male, 5 Meter, Black Audio Cable #10729   
11398                                              TP-Link TL-SG1008D 8 Port Gigabit Switch   
11467                                        Verbex VT-801D Conference Delegates Microphone   
11494         UGREEN HD104 (10111) HDMI 1.4 Male to Male, 15 Meter, Black Cable #10111 (4K)   
11511  Western Digital My Passport 4TB USB 3.2 Gen 1 Black External HDD #WDBPKJ0040BBK-WESN   

                                                                           seller_product_url  \
11354            https://www.ryans.com/microsoft-visual-studio-pro-2019-sngl-olp-nl-c5e-01380   
11385                                    https://www.ryans.com/ugreen-av119-10729-audio-c

In [86]:
# Fill up the category_name and category_slug with 'uncategorized' for any nulls
df['category_name'] = df['category_name'].fillna('Uncategorized')
df['category_slug'] = df['category_slug'].fillna('uncategorized')

## Populate seller_slug

In [87]:
# Populate seller_slug column with 'ryans'
df['seller_slug'] = 'ryans'

## Inspect and fix current_price and original_price

In [88]:
# Inspect and fix current_price and original_price columns

# Show the rows where current_price is not a valid number
invalid_current_price = df[~df['current_price'].apply(lambda x: isinstance(x, (int, float)) or (isinstance(x, str) and x.replace('.', '', 1).isdigit()))]
print("\nRows with invalid current_price:")
print(invalid_current_price[['current_price']])

# Show the rows where original_price is not a valid number
invalid_original_price = df[~df['original_price'].apply(lambda x: isinstance(x, (int, float)) or (isinstance(x, str) and x.replace('.', '', 1).isdigit()))]
print("\nRows with invalid original_price:")
print(invalid_original_price[['original_price']])

# Show the rows where current_price is null
null_current_price = df[df['current_price'].isnull()]
print("\nRows with null current_price:")
print(null_current_price[['product_name', 'seller_product_url', 'current_price']])

# Show the rows where original_price is null
null_original_price = df[df['original_price'].isnull()]
print("\nRows with null original_price:")
print(null_original_price[['product_name', 'seller_product_url', 'original_price']])


Rows with invalid current_price:
Empty DataFrame
Columns: [current_price]
Index: []

Rows with invalid original_price:
Empty DataFrame
Columns: [original_price]
Index: []

Rows with null current_price:
                                                                                                                  product_name  \
11413                                                   Wacom One M CTC-6110WL Medium White Graphics Drawing Tablet #CTC6110WL   
11461  QNAP TL-D800S 8 Bays Mini SAS (SFF-8088) Input & Output / SATA III Expansion Unit for Network Storage (2 Year Warranty)   

                                                                      seller_product_url  \
11413  https://www.ryans.com/wacom-one-m-ctc-6110wl-medium-white-graphics-drawing-tablet   
11461                                https://www.ryans.com/qnap-tl-d800s-network-storage   

       current_price  
11413            NaN  
11461            NaN  

Rows with null original_price:
                            

In [89]:
# Make the current_price to 0 and in_stock to "NO" if current_price is NaN
df.loc[df['current_price'].isnull(), 'current_price'] = 0.0
df.loc[df['current_price'] == 0, 'in_stock'] = "NO"

# Make the original_price to 0 and in_stock to "NO" if original_price is NaN
df.loc[df['original_price'].isnull(), 'original_price'] = 0.0
df.loc[df['original_price'] == 0, 'in_stock'] = "NO"

# Check the reflected changes, show the data info again to confirm the price columns are now numeric and have no nulls
print("\nData info after cleaning price columns:")
print(df.info())


Data info after cleaning price columns:
<class 'pandas.DataFrame'>
Index: 11364 entries, 0 to 11535
Data columns (total 35 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   category_slug            11364 non-null  str    
 1   product_name             11364 non-null  str    
 2   product_slug             11364 non-null  str    
 3   seller_slug              11364 non-null  str    
 4   current_price            11364 non-null  float64
 5   seller_product_url       11364 non-null  str    
 6   brand_slug               11358 non-null  str    
 7   brand_name               11358 non-null  str    
 8   product_description      11363 non-null  str    
 9   model                    0 non-null      float64
 10  sku                      0 non-null      float64
 11  primary_image_url        11355 non-null  str    
 12  image_urls               0 non-null      float64
 13  specifications           0 non-null      float64
 1

## Inspect brand_name and brand_slug

In [90]:
# Inspect brand_name and brand_slug
print("\nUnique values in 'brand_name':")
pd.set_option('display.max_rows', None)
print(df['brand_name'].value_counts())
pd.reset_option('display.max_rows')


Unique values in 'brand_name':
brand_name
All Brands                    1290
Mobile Accessories             363
ASUS                           274
USB Cable                      234
Speaker                        185
UGREEN                         176
Havit                          172
HP                             155
Gigabyte                       143
Apple                          137
Type-C Converter               136
MSI                            135
Canon                          135
PC Power                       130
Hikvision                      127
Lenovo                         112
Cable / Converter / HUB        111
Panasonic                      110
Samsung                        110
A4TECH                         103
Keyboard                       103
QNAP                           103
ONIKUMA                        100
Value Top                       99
Philips                         96
Dahua                           94
TP-Link                         91
Singer      

In [91]:
# Replace the null and where category_name == 'brand_name' with First word of the product_name as brand_name and its slug as brand_slug
def extract_brand(row):
    unusual_brand_names = [
    "USB Cable", "Speaker", "Type-C Converter", "Keyboard", "Audio Cable", 
    "USB Converter", "Network Accessories", "Scanner", "Document Printer", 
    "Label Printers", "POS Printers", "Keyboard and Mouse Combo", 
    "Studio Microphone", "POS and Label Supplies", "Power", 
    "Conference System", "Bag", "Audio Converter", "Smart Door Bell", 
    "Graphics Tablet PC", "Gaming Laptop", "Digital Display", "Smart", 
    "Gaming Monitor", "Home Theater Systems", "Printer Paper", "Fridge", 
    "Virtual Reality (VR)", "POS System", "Kitchen", "Armor", "Touch", 
    "(Bundle", "PC", "Power Guard", "Tripod", "Cooler", "Compact Camera Accessories", 
    "Lightning Converter", "Smart Lock", "Sound System Accessories", 
    "Desktop Accessories", "All Monitor", "Compatible", "General", 
    "TV Accessories", "Digital Signature Pad", "Tablet Accessories", 
    "POS Accessories", "Digital", "June Server", "Audio Interface", 
    "Gaming Desktop Component", "Gaming", "Gaming Network Router", "Super", 
    "Television", "Internal SSD", "Microsoft Accessories", "All Laptop", 
    "Video Camera", "Flash and Ring Light", "CR-80", "Ribbon", "3-Pin", 
    "Earphone", "FAN", "Studio Equipments", "Studio Headphone", "A4", 
    "Umbrella", "TP", "Accessories", "Mixer", "The", "Gaming Component", 
    "Death", "A", "Exclusive", "Gadget", "HDMI", "VGA", "Weighing Scale", 
    "Print Head", "Apple Tablet PC", "Home Care", "Space", "Projector", 
    "Laser Printer", "Amplifier", "Casing", "PVC", "Processor", 
    "Desktop and Server", "Pen Drive", "UPS", "(Single)", 
    "Server Motherboard", "Server Processor", "Server Component"
]
    if pd.isnull(row['brand_name']) or row['brand_name'] in unusual_brand_names:
        if isinstance(row['product_name'], str) and len(row['product_name'].split()) > 0:
            return row['product_name'].split()[0]  # Take the first word of product_name as brand
    return row['brand_name']
df['brand_name'] = df.apply(extract_brand, axis=1)

# Generate brand_slug from brand_name
df['brand_slug'] = df['brand_name'].apply(slugify)

# Check the changes in brand_name and brand_slug
print("\nUnique values in 'brand_name' after extraction:")
pd.set_option('display.max_rows', None)
print(df['brand_name'].value_counts())
pd.reset_option('display.max_rows')

print("\nUnique values in 'brand_slug' after extraction:")
pd.set_option('display.max_rows', None)
print(df['brand_slug'].value_counts())


Unique values in 'brand_name' after extraction:
brand_name
All Brands                 1290
Mobile Accessories          363
UGREEN                      356
ASUS                        286
Havit                       200
HP                          169
Vention                     155
Gigabyte                    147
Canon                       144
Apple                       141
Onten                       137
MSI                         136
Lenovo                      131
Yuanxin                     130
PC Power                    130
Hikvision                   129
Samsung                     119
A4TECH                      112
Logitech                    112
Cable / Converter / HUB     111
Panasonic                   111
WiWU                        103
TP-Link                     103
QNAP                        103
ONIKUMA                     101
Value Top                    99
JBL                          98
Philips                      97
Singer                       96
Dahua       

In [92]:
# Check the data after all cleaning steps
print("\nData after cleaning:")
print(df.info())


Data after cleaning:
<class 'pandas.DataFrame'>
Index: 11364 entries, 0 to 11535
Data columns (total 35 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   category_slug            11364 non-null  str    
 1   product_name             11364 non-null  str    
 2   product_slug             11364 non-null  str    
 3   seller_slug              11364 non-null  str    
 4   current_price            11364 non-null  float64
 5   seller_product_url       11364 non-null  str    
 6   brand_slug               11364 non-null  str    
 7   brand_name               11364 non-null  str    
 8   product_description      11363 non-null  str    
 9   model                    0 non-null      float64
 10  sku                      0 non-null      float64
 11  primary_image_url        11355 non-null  str    
 12  image_urls               0 non-null      float64
 13  specifications           0 non-null      float64
 14  attributes      

In [93]:
# Check the number of 'in_stock' unique values
print("\nUnique 'in_stock' values:")
print(df['in_stock'].nunique())
print(df['in_stock'].value_counts())


Unique 'in_stock' values:
2
in_stock
Yes    11362
NO         2
Name: count, dtype: int64


In [94]:
# Check the review_count column to see if there are any non-numeric values that need to be cleaned
print("\nReview count unique values:")
print(df['review_count'].nunique())
print(df['review_count'].value_counts())


Review count unique values:
21
review_count
0     3881
10    2261
1     1001
11     616
9      541
2      460
8      405
12     334
3      277
7      271
4      221
13     179
6      167
5      166
20     156
14     112
15     100
17      63
16      58
18      50
19      45
Name: count, dtype: int64


In [95]:
# Save the csv file to the csvs folder with the name 'StarTech_products_cleaned.csv'
cleaned_data_file = f"{seller_name}_products_cleaned.csv"
cleaned_data_file_path = seller_data_dir / cleaned_data_file
df.to_csv(cleaned_data_file_path, index=False)
print(f"\nCleaned data saved to {cleaned_data_file_path}")


Cleaned data saved to /home/abrar/Freelance/DaamDekho/Crawler/data/csvs/Ryans_products_cleaned.csv


In [96]:
# Put the cleaned data into 2500 rows per csv file into the uploads directory
chunk_size = 2500
for i in range(0, len(df), chunk_size):
    chunk = df[i:i+chunk_size]
    chunk_file_path = upload_dir / f"{seller_name}_products_cleaned_part_{i//chunk_size + 1}.csv"
    chunk.to_csv(chunk_file_path, index=False)
    print(f"Saved chunk {i//chunk_size + 1} to {chunk_file_path}")

Saved chunk 1 to /home/abrar/Freelance/DaamDekho/Crawler/data/csvs/uploads/Ryans/Ryans_products_cleaned_part_1.csv
Saved chunk 2 to /home/abrar/Freelance/DaamDekho/Crawler/data/csvs/uploads/Ryans/Ryans_products_cleaned_part_2.csv
Saved chunk 3 to /home/abrar/Freelance/DaamDekho/Crawler/data/csvs/uploads/Ryans/Ryans_products_cleaned_part_3.csv
Saved chunk 4 to /home/abrar/Freelance/DaamDekho/Crawler/data/csvs/uploads/Ryans/Ryans_products_cleaned_part_4.csv
Saved chunk 5 to /home/abrar/Freelance/DaamDekho/Crawler/data/csvs/uploads/Ryans/Ryans_products_cleaned_part_5.csv
